# Notebook 02 — Depth-dependent unprecedented split

Goal: reproduce the Fisher's-exact-style permutation test for the
3/3 surface-thermocline vs 0/2 deep split in the five testable
Caesar-2021 proxies.

Inputs: hardcoded proxy table (depth + |z|_max + group).
Outputs:
  - p_exhaustive (= 0.10, deterministic)
  - p_monte_carlo (≈ 0.097, 10,000 permutations)

Acceptance: p_exhaustive == 0.10, p_monte_carlo in [0.08, 0.11].

In [ ]:
from itertools import combinations
import numpy as np

proxies = [
    ('Thornalley 2018 Tsub',         400.0,  19.7, 'surface'),
    ('Spooner 2020 T. quinqueloba',  100.0,   4.8, 'surface'),
    ('Rahmstorf 2015 multi-proxy',    50.0,  22.1, 'surface'),
    ('Thornalley 2018 sortable silt', 2000.0,   0.8, 'deep'),
    ('Osmann 2019 MAS productivity',  3500.0,   0.0, 'deep'),
]

n = len(proxies)
n_unprec = sum(1 for *_, z, _ in proxies if z > 3)
surf_idx = {i for i, (_, _, _, g) in enumerate(proxies) if g == 'surface'}

# Exhaustive: C(n, n_unprec)
total = 0
extreme = 0
for combo in combinations(range(n), n_unprec):
    total += 1
    n_surf = len(set(combo) & surf_idx)
    if n_surf == n_unprec:   # all unprec assigned to surface
        extreme += 1
p_exhaustive = extreme / total
print(f'Exhaustive enumeration: {extreme}/{total} = {p_exhaustive:.4f}')

# Monte Carlo cross-check
rng = np.random.default_rng(42)
B = 10_000
extreme_mc = 0
for _ in range(B):
    labels = rng.permutation(n)
    n_surf = len(set(labels[:n_unprec]) & surf_idx)
    if n_surf == n_unprec:
        extreme_mc += 1
print(f'Monte Carlo ({B} perms): {extreme_mc}/{B} = {extreme_mc/B:.4f}')

assert p_exhaustive == 0.1, 'p_exhaustive should be 0.10'
assert 0.08 <= extreme_mc / B <= 0.11, 'p_MC should be in [0.08, 0.11]'
print('PASS')